<a href="https://colab.research.google.com/github/Lucaaa31/Anomaly-Segmentation/blob/master/notebooks/Step8_with_Temperature.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Step 8 Anomaly Segmentation
---


# Settings


In [1]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/Anomaly-Segmentation
#!git pull origin master

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/.shortcut-targets-by-id/1Pz7ReDC4oIzyB7KLXs9SnUMvmnDqYbyB/Anomaly-Segmentation


## Dependencies

In [ ]:
!pip install --upgrade-strategy only-if-needed -r requirements.txt

## Imports and random seeds

In [3]:
import os
import glob
import yaml
import random
import warnings
import importlib
from torch.amp.autocast_mode import autocast
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from torchvision.transforms import Compose, Resize, ToTensor
from lightning import seed_everything
from huggingface_hub import hf_hub_download
from huggingface_hub.utils import RepositoryNotFoundError
from ood_metrics import fpr_at_95_tpr
from sklearn.metrics import average_precision_score
import matplotlib.pyplot as plt
import torchvision.transforms.functional as F_tf
import torchvision.transforms as T
from torch.utils.data import TensorDataset, DataLoader


# Utils
from utils.build import build_model_and_data
from data.load_eomt.cityscapes_semantic import CityscapesSemantic
from utils.temperature_scaling import ModelWithTemperature


seed_everything(42, verbose=False)
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = True

---
# Evaluation
## Configuration


In [34]:
dataset_to_use = "RoadAnomaly21"  # @param ["fs_static", "RoadAnomaly21", "RoadAnomaly", "FS_LostFount_full", "RoadObsticle21"]
methods = "RbA"               # @param ["MSP", "max_logit", "max_entropy", "RbA"]

project_root    = "/content/drive/MyDrive/Anomaly-Segmentation"

data_path       = project_root + f"/dataset/Anomaly_Validation_Dataset/{dataset_to_use}"
input_pattern   = data_path + "/images/*"

model_type =  "cityscapes" # @param ["coco", "cityscapes", "finetuned"]

local_ckpt = project_root + f"/models/eomt_eval/eomt_{model_type}.bin"

config_path = project_root + f"/configs/dinov2/{model_type}/{"panoptic" if model_type in ("coco", "finetuned") else "semantic"}/eomt_base_640{"_2x" if model_type in ("coco", "finetuned") else "" }.yaml"

temperature = "none" # @param ["none","0.5","0.75","1.1","best"]




device = "cuda" if torch.cuda.is_available() else "cpu"


class Args:
    def __init__(self):
        self.input = input_pattern
        self.datadir = data_path
        self.method = methods

args = Args()
#print(type(temperature))
# print(f"Selected parameters: {vars(args)}\n")
# print(f"Weight's path: {local_ckpt}")
# print(f"Config path: {config_path}")


We build the model and wrap it in the ModelWithTemperature Class:

In [26]:
print(f"Building the eomt-{model_type}")
model, meta = build_model_and_data(config_path, local_ckpt, data_path, device, setup_data=False)
print(f"  num_classes={meta.num_classes}  img_size={meta.img_size}")
NUM_CLASSES = meta.num_classes
IMG_SIZE = meta.img_size
temperature_model = ModelWithTemperature(model)

Building the eomt-cityscapes


/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'network' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['network'])`.


  num_classes=19  img_size=(1024, 1024)


## Temperature

In [ ]:
def find_best_temperature():
    all_images_for_calibration = []
    all_pixel_labels_for_calibration = []

    for path in glob.glob(os.path.expanduser(str(args.input))):
        raw_image = Image.open(path).convert('RGB')
        images = input_transform(raw_image)

        pathGT = path.replace("images", "labels_masks")
        if "RoadObsticle21" in pathGT:
            pathGT = pathGT.replace("webp", "png")
        if "fs_static" in pathGT:
            pathGT = pathGT.replace("jpg", "png")
        if "RoadAnomaly" in pathGT:
            pathGT = pathGT.replace("jpg", "png")

        raw_target = Image.open(pathGT).convert('L')
        target_np = target_transform(raw_target)

        all_images_for_calibration.append(images)
        all_pixel_labels_for_calibration.append(torch.from_numpy(target_np).long().to(device))


    concatenated_images = torch.cat(all_images_for_calibration, dim=0)
    concatenated_pixel_labels = torch.cat(all_pixel_labels_for_calibration, dim=0)

    calibration_dataset_tensors = TensorDataset(concatenated_images, concatenated_pixel_labels)
    calibration_loader = DataLoader(calibration_dataset_tensors, batch_size=1, shuffle=False)

    temperature_model.set_temperature(calibration_loader)
    return



In [ ]:
if temperature == "none":
    pass
elif temperature == "best":
    find_best_temperature(temperature_model)
else:
    temperature_model.temperature = torch.nn.Parameter(torch.tensor([float(temperature)]))


## Utils methods


In [13]:
def infer_panoptic(img, target):
    with torch.no_grad(), autocast(dtype=torch.float16, device_type="cuda"):
        imgs = [img.to(device)]
        img_sizes = [img.shape[-2:] for img in imgs]


        transformed_imgs = model.resize_and_pad_imgs_instance_panoptic(imgs)
        mask_logits_per_layer, class_logits_per_layer = model(transformed_imgs)

        mask_logits = F.interpolate(
            mask_logits_per_layer[-1], model.img_size, mode="bilinear"
        )
        mask_logits = model.revert_resize_and_pad_logits_instance_panoptic(
            mask_logits, img_sizes
        )



        preds = model.to_per_pixel_preds_panoptic(
            mask_logits,
            class_logits_per_layer[-1],
            model.stuff_classes,
            model.mask_thresh,
            model.overlap_thresh,
        )[0].cpu()


    pred = preds.numpy()
    sem_pred, inst_pred = pred[..., 0], pred[..., 1]

    target_seg = model.to_per_pixel_targets_panoptic([target])[0].cpu().numpy()
    sem_target, inst_target = target_seg[..., 0], target_seg[..., 1]

    # Compute the raw logits
    cls_logits = class_logits_per_layer[-1][0][:, :NUM_CLASSES + 1] # Use actual NUM_CLASSES + 1 for background
    masks_probs = torch.sigmoid(mask_logits[0])

    H, W = masks_probs.shape[1], masks_probs.shape[2]


    num_queries = cls_logits.shape[0]
    num_output_classes = cls_logits.shape[1] # This will be NUM_CLASSES + 1

    semantic_logits = torch.mm(cls_logits.t(), masks_probs.view(num_queries, -1).to(cls_logits.dtype)).view(num_output_classes, H, W)

    return sem_pred, inst_pred, sem_target, inst_target, semantic_logits


In [7]:
def draw_black_border(sem, inst, mapping):
    h, w = sem.shape
    out = np.zeros((h, w, 3))
    for s in np.unique(sem):
        out[sem == s] = mapping[s]

    combined = sem.astype(np.int64) * 100000 + inst.astype(np.int64)
    border = np.zeros((h, w), dtype=bool)
    border[1:, :] |= combined[1:, :] != combined[:-1, :]
    border[:-1, :] |= combined[1:, :] != combined[:-1, :]
    border[:, 1:] |= combined[:, 1:] != combined[:, :-1]
    border[:, :-1] |= combined[:, 1:] != combined[:, :-1]
    out[border] = 0
    return out


def plot_panoptic_results(img, sem_pred, inst_pred, sem_target, inst_target):
    all_ids = np.union1d(np.unique(sem_pred), np.unique(sem_target))
    mapping = {
        s: (
            [0, 0, 0]
            if s == -1 or s == model.num_classes
            else plt.cm.hsv(i / len(all_ids))[:3]
        )
        for i, s in enumerate(all_ids)
    }

    vis_pred = draw_black_border(sem_pred, inst_pred, mapping)
    vis_target = draw_black_border(sem_target, inst_target, mapping)

    img_np = (
        img.cpu().numpy().transpose(1, 2, 0) if img.dim() == 3 else img.cpu().numpy()
    )

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(img_np)
    axes[0].set_title("Input")
    axes[1].imshow(vis_pred)
    axes[1].set_title("Prediction")
    axes[2].imshow(vis_target)
    axes[2].set_title("Target")

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

## Post hoc methods

In [16]:
def get_msp(logits):

    probs = F.softmax(logits, dim=0)  # (C, H, W), softmax over classes (dim=0)
    msp = probs.max(dim=0).values  # (H, W)


    num_classes = logits.shape[0] # Number of classes is dim=0
    min_possible_msp = 1.0 / num_classes


    msp_anomaly = (1.0 - msp) / (1.0 - min_possible_msp)

    return msp_anomaly.cpu().numpy().astype("float32")


def get_maxlogit(logits):

    max_logit = logits.max(dim=0).values  # (H, W)


    anomaly_score = -max_logit

    min_val = anomaly_score.min()
    max_val = anomaly_score.max()

    if max_val > min_val:
        anomaly_score = (anomaly_score - min_val) / (max_val - min_val)
    else:
        anomaly_score = torch.zeros_like(anomaly_score)

    return anomaly_score.cpu().numpy().astype("float32")


def get_entropy(logits):
    probs = F.softmax(logits, dim=0)  # (C, H, W), softmax over classes (dim=0)
    num_classes = probs.shape[0] # Number of classes is dim=0
    device = logits.device

    entropy = -(probs * (probs + 1e-7).log()).sum(dim=0)  # sum over classes (dim=0)

    denom = torch.log(
        torch.tensor(num_classes, dtype=torch.float32, device=device)
    )
    norm_entropy = entropy / denom

    return norm_entropy.cpu().numpy().astype("float32")


def get_Rba(logits):

    top_logits, _ = torch.topk(logits, k=2, dim=0) # logits is (C, H, W), topk over dim=0
    max_logit = top_logits[0]
    second_max_logit = top_logits[1]

    margin = max_logit - second_max_logit

    anomaly_score = -margin

    min_val = anomaly_score.min()
    max_val = anomaly_score.max()

    if max_val > min_val:
        anomaly_score = (anomaly_score - min_val) / (max_val - min_val)
    else:
        anomaly_score = torch.zeros_like(anomaly_score)

    return anomaly_score.cpu().numpy().astype("float32")


## Preprocessing

In [9]:
def input_transform(img):
    img_resized = F_tf.resize(img, IMG_SIZE, interpolation=T.InterpolationMode.BILINEAR)
    img_tensor = F_tf.to_tensor(img_resized)
    return (img_tensor * 255).to(torch.uint8).to(device)

def target_transform(img):
    img_resized = F_tf.resize(img, IMG_SIZE, interpolation=T.InterpolationMode.NEAREST)
    return np.array(img_resized)

## Pipeline

In [35]:
anomaly_score_list = []
ood_gts_list = []



for path in glob.glob(os.path.expanduser(str(args.input))):
    raw_image = Image.open(path).convert('RGB')
    images = input_transform(raw_image)

    pathGT = path.replace("images", "labels_masks")

    if "RoadObsticle21" in pathGT:
        pathGT = pathGT.replace("webp", "png")
    if "fs_static" in pathGT:
        pathGT = pathGT.replace("jpg", "png")
    if "RoadAnomaly" in pathGT:
        pathGT = pathGT.replace("jpg", "png")



    raw_target = Image.open(pathGT).convert('L')
    target_np = target_transform(raw_target)


    unique_ids = np.unique(target_np)
    unique_ids = unique_ids[unique_ids != 0]

    masks = []
    labels = []
    for s_id in unique_ids:
        binary_mask = (target_np == s_id)
        masks.append(binary_mask)
        labels.append(s_id)

    if len(masks) > 0:
        target_dict = {
            "masks": torch.from_numpy(np.stack(masks)).bool().to(device),
            "labels": torch.from_numpy(np.array(labels)).long().to(device)
        }
    else:
        target_dict = {
            "masks": torch.zeros((0, target_np.shape[0], target_np.shape[1]), dtype=torch.bool).to(device),
            "labels": torch.zeros((0,), dtype=torch.long).to(device)
        }



    sem_pred, inst_pred, sem_target, inst_target, result_logits = infer_panoptic(images, target_dict)
    #plot_panoptic_results(images, sem_pred, inst_pred, sem_target, inst_target)



    match args.method:
        case "MSP":
            anomaly_result = get_msp(result_logits)
        case "max_logit":
            anomaly_result = get_maxlogit(result_logits)
        case "max_entropy":
            anomaly_result = get_entropy(result_logits)
        case "RbA":
            anomaly_result = get_Rba(result_logits)

    # print(f"Anomaly result for the {args.method}: {anomaly_result}")

    pathGT = path.replace("images", "labels_masks")
    if "RoadObsticle21" in pathGT:
        pathGT = pathGT.replace("webp", "png")
    if "fs_static" in pathGT:
        pathGT = pathGT.replace("jpg", "png")
    if "RoadAnomaly" in pathGT:
        pathGT = pathGT.replace("jpg", "png")


    ood_gts = ood_gts = target_np.copy()

    if "RoadAnomaly" in pathGT:
        ood_gts = np.where((ood_gts==2), 1, ood_gts)
    if "LostAndFound" in pathGT:
        ood_gts = np.where((ood_gts==0), 255, ood_gts)
        ood_gts = np.where((ood_gts==1), 0, ood_gts)
        ood_gts = np.where((ood_gts>1) & (ood_gts<201), 1, ood_gts)

    if "Streethazard" in pathGT:
        ood_gts = np.where((ood_gts==14), 255, ood_gts)
        ood_gts = np.where((ood_gts<20), 0, ood_gts)
        ood_gts = np.where((ood_gts==255), 1, ood_gts)

    if 1 not in np.unique(ood_gts):
        continue
    else:
            ood_gts_list.append(ood_gts)
            anomaly_score_list.append(anomaly_result)
    del result_logits, anomaly_result, ood_gts
    torch.cuda.empty_cache()


/tmp/ipykernel_29928/1591009150.py:2: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  with torch.no_grad(), autocast(dtype=torch.float16, device_type="cuda"):


## Save of the Results

In [ ]:
if not os.path.exists('results/Step8.txt'):
    open('results/Step8.txt', 'w').close()
file = open('results/Step8.txt', 'a')

file.write("\n")

ood_gts = np.array(ood_gts_list)
anomaly_scores = np.array(anomaly_score_list)

ood_mask = (ood_gts == 1)
ind_mask = (ood_gts == 0)

ood_out = anomaly_scores[ood_mask]
ind_out = anomaly_scores[ind_mask]

ood_label = np.ones(len(ood_out))
ind_label = np.zeros(len(ind_out))

val_out = np.concatenate((ind_out, ood_out))
val_label = np.concatenate((ind_label, ood_label))

print("Labels:", val_label)
print("Scores:", val_out)

prc_auc = average_precision_score(val_label, val_out)
fpr = fpr_at_95_tpr(val_out, val_label)

print(f'AUPRC score: {prc_auc * 100.0:.4f}%')
print(f'FPR@TPR95: {fpr * 100.0:.4f}%')

header = f"Method: {args.method} | Dataset: {dataset_to_use} | Model: {model_type}\n"
metrics = f"AUPRC: {prc_auc * 100.0:.5f}% | FPR@TPR95: {fpr * 100.0:.5f}%\n"
divider = "=" * 100 + "\n"

file.write(header + metrics + divider)
file.close()